In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🤖 Advanced Machine Learning Patterns & Techniques Guide

**Deep dive into production ML systems, transfer learning, ensemble methods, and AutoML**

This guide covers:
- Transfer learning and fine-tuning strategies
- Ensemble methods and stacking
- AutoML and hyperparameter optimization
- Feature engineering at scale
- Model serving and inference optimization
- A/B testing ML models
- Drift detection and model monitoring
- MLOps best practices
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Transfer Learning & Fine-tuning

### Pre-trained Model Fine-tuning
```python
import torch
import torch.nn as nn
from transformers import BertForSequenceClassification, BertTokenizer, AdamW
from torch.utils.data import DataLoader, TensorDataset

class TransferLearningFineTuner:
    def __init__(self, model_name: str = "bert-base-uncased"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2
        )
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
    
    def prepare_data(self, texts: list, labels: list, max_length: int = 128):
        """Tokenize and prepare data"""
        encodings = self.tokenizer(
            texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )
        
        dataset = TensorDataset(
            encodings['input_ids'],
            encodings['attention_mask'],
            torch.tensor(labels)
        )
        
        return dataset
    
    def fine_tune(self, train_loader, epochs: int = 3, learning_rate: float = 2e-5):
        """Fine-tune model on downstream task"""
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)
        
        for epoch in range(epochs):
            self.model.train()
            total_loss = 0
            
            for batch in train_loader:
                input_ids, attention_mask, labels = batch
                input_ids = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                labels = labels.to(self.device)
                
                # Forward pass
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                loss = outputs.loss
                total_loss += loss.item()
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            avg_loss = total_loss / len(train_loader)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    def predict(self, texts: list):
        """Make predictions"""
        self.model.eval()
        
        encodings = self.tokenizer(
            texts,
            max_length=128,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )
        
        with torch.no_grad():
            outputs = self.model(
                input_ids=encodings['input_ids'].to(self.device),
                attention_mask=encodings['attention_mask'].to(self.device)
            )
        
        predictions = torch.softmax(outputs.logits, dim=1)
        return predictions.cpu().numpy()

# Usage
finetuner = TransferLearningFineTuner()
train_dataset = finetuner.prepare_data(texts, labels)
train_loader = DataLoader(train_dataset, batch_size=32)
finetuner.fine_tune(train_loader)
```

### Domain Adaptation
```python
import numpy as np
from sklearn.preprocessing import StandardScaler

class DomainAdaptation:
    """Adapt model from source to target domain"""
    
    def __init__(self):
        self.source_mean = None
        self.source_std = None
    
    def instance_weighting(self, source_data, target_data):
        """Weight source instances by similarity to target"""
        # Calculate importance weights
        source_density = self._estimate_density(source_data)
        target_density = self._estimate_density(target_data)
        
        # Importance = P(target) / P(source)
        weights = np.clip(target_density / source_density, 0.1, 10)
        return weights
    
    def feature_alignment(self, source_features, target_features):
        """Align feature distributions"""
        # Normalize source to target domain
        target_mean = np.mean(target_features, axis=0)
        target_std = np.std(target_features, axis=0)
        
        source_mean = np.mean(source_features, axis=0)
        source_std = np.std(source_features, axis=0)
        
        # Standardize and transform
        normalized = (source_features - source_mean) / source_std
        aligned = normalized * target_std + target_mean
        
        return aligned
    
    def _estimate_density(self, data):
        """KDE-based density estimation"""
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(data.T)
        return kde(data.T)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Ensemble Methods & Stacking

### Voting Ensemble
```python
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

class EnsembleModel:
    def __init__(self):
        # Create base learners
        lr = LogisticRegression(max_iter=1000)
        rf = RandomForestClassifier(n_estimators=100)
        gb = GradientBoostingClassifier(n_estimators=100)
        
        # Voting ensemble
        self.ensemble = VotingClassifier(
            estimators=[('lr', lr), ('rf', rf), ('gb', gb)],
            voting='soft'  # Use probabilities
        )
    
    def train(self, X_train, y_train):
        """Train ensemble"""
        self.ensemble.fit(X_train, y_train)
    
    def predict(self, X_test):
        """Get predictions"""
        return self.ensemble.predict(X_test)
    
    def predict_proba(self, X_test):
        """Get probability predictions"""
        return self.ensemble.predict_proba(X_test)
```

### Stacking Meta-Learner
```python
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import Ridge

class StackingEnsemble:
    """Stack base learners with meta-learner"""
    
    def __init__(self, base_learners, meta_learner=None):
        self.base_learners = base_learners
        self.meta_learner = meta_learner or Ridge()
        self.trained_base_learners = []
    
    def train(self, X_train, y_train):
        """Train base learners and meta-learner"""
        # Generate meta-features using cross-validation
        meta_features = []
        
        for learner in self.base_learners:
            # Cross-validation predictions
            meta_pred = cross_val_predict(
                learner,
                X_train,
                y_train,
                cv=5,
                method='predict_proba'
            )
            meta_features.append(meta_pred)
        
        # Stack predictions
        meta_X = np.column_stack(meta_features)
        
        # Train base learners on full data
        for learner in self.base_learners:
            learner.fit(X_train, y_train)
            self.trained_base_learners.append(learner)
        
        # Train meta-learner
        self.meta_learner.fit(meta_X, y_train)
    
    def predict(self, X_test):
        """Predictions using stacked ensemble"""
        meta_features = []
        
        for learner in self.trained_base_learners:
            meta_pred = learner.predict_proba(X_test)
            meta_features.append(meta_pred)
        
        meta_X = np.column_stack(meta_features)
        return self.meta_learner.predict(meta_X)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. AutoML & Hyperparameter Optimization

### Bayesian Optimization
```python
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

class BayesianHyperparameterOptimizer:
    """Optimize hyperparameters using Bayesian optimization"""
    
    def __init__(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
    
    def objective(self, params):
        """Objective function to minimize"""
        n_estimators, max_depth, min_samples_split = params
        
        model = RandomForestClassifier(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth),
            min_samples_split=int(min_samples_split),
            random_state=42
        )
        
        # Negative because we want to minimize
        score = -cross_val_score(
            model,
            self.X_train,
            self.y_train,
            cv=5,
            scoring='accuracy'
        ).mean()
        
        return score
    
    def optimize(self, n_calls: int = 20):
        """Run Bayesian optimization"""
        space = [
            Integer(10, 300, name='n_estimators'),
            Integer(5, 50, name='max_depth'),
            Integer(2, 20, name='min_samples_split')
        ]
        
        result = gp_minimize(
            self.objective,
            space,
            n_calls=n_calls,
            random_state=42,
            n_initial_points=5
        )
        
        return result.x, -result.fun

# Usage
optimizer = BayesianHyperparameterOptimizer(X_train, y_train)
best_params, best_score = optimizer.optimize()
```

### Grid Search with Cross-Validation
```python
from sklearn.model_selection import GridSearchCV

class ModelTuner:
    def __init__(self, model, param_grid):
        self.model = model
        self.param_grid = param_grid
    
    def tune(self, X_train, y_train, cv: int = 5):
        """Grid search with cross-validation"""
        grid_search = GridSearchCV(
            self.model,
            self.param_grid,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X_train, y_train)
        
        return {
            'best_params': grid_search.best_params_,
            'best_score': grid_search.best_score_,
            'cv_results': grid_search.cv_results_
        }
```
</VSCode.Cell>

<MySCode.Cell language="markdown">
## 4. Model Serving & Inference Optimization

### ONNX Model Export
```python
import torch
import onnx
from onnx import optimizer

class ModelExporter:
    """Export PyTorch models to ONNX"""
    
    @staticmethod
    def export_to_onnx(model, example_input, output_path: str):
        """Export model to ONNX format"""
        torch.onnx.export(
            model,
            example_input,
            output_path,
            export_params=True,
            opset_version=14,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            verbose=False
        )
        
        # Optimize ONNX model
        onnx_model = onnx.load(output_path)
        optimized_model = optimizer.optimize(onnx_model)
        onnx.save(optimized_model, output_path)
        
        return output_path

# Usage
exporter = ModelExporter()
exporter.export_to_onnx(model, torch.randn(1, 3, 224, 224), "model.onnx")
```

### Model Quantization
```python
import torch.quantization as tq

class ModelQuantizer:
    """Quantize models for faster inference"""
    
    @staticmethod
    def quantize_dynamic(model):
        """Dynamic quantization (faster, less accurate)"""
        quantized_model = tq.quantize_dynamic(
            model,
            {torch.nn.Linear},
            dtype=torch.qint8
        )
        return quantized_model
    
    @staticmethod
    def quantize_static(model, calibration_loader):
        """Static quantization (slower to prepare, best accuracy)"""
        model.qconfig = tq.get_default_qconfig('fbgemm')
        tq.prepare(model, inplace=True)
        
        # Calibrate on representative data
        with torch.no_grad():
            for images, labels in calibration_loader:
                model(images)
        
        tq.convert(model, inplace=True)
        return model
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. ML Best Practices Checklist

✅ **Model Development**
- [ ] Baseline established before complex models
- [ ] Cross-validation used (k-fold, stratified)
- [ ] Train/validation/test split proper
- [ ] Class imbalance handled
- [ ] Feature scaling applied consistently

✅ **Ensemble Methods**
- [ ] Diverse base learners selected
- [ ] Correlation between learners checked
- [ ] Stacking validation set used
- [ ] Ensemble diversity measured
- [ ] Interpretability maintained

✅ **Hyperparameter Tuning**
- [ ] Search space defined properly
- [ ] Early stopping implemented
- [ ] Learning curves analyzed
- [ ] Overfitting monitored
- [ ] Reproducibility ensured (random seed)

✅ **Model Serving**
- [ ] Model serialization tested
- [ ] Inference latency < SLA
- [ ] Input validation implemented
- [ ] Error handling robust
- [ ] Monitoring in place

✅ **Production Readiness**
- [ ] Model versioning implemented
- [ ] A/B testing framework ready
- [ ] Rollback procedure documented
- [ ] Monitoring dashboards set up
- [ ] Drift detection configured
</MySCode.Cell>
```